# Test discriminante: SAURIA preload A → Gather overwrite A → SAURIA senza reload A

Il test avvelena l'IFMAP originale in DRAM, esegue una prima SAURIA standard,
fa sovrascrivere SRAM A dal gather e avvia una seconda SAURIA con il nuovo
bit `control_regs[21][26]`. Solo A viene saltata; B/C, calcolo e writeback
restano nel percorso standard.

In [1]:
from pathlib import Path
import json, subprocess, sys

def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "RTL").is_dir() and (candidate / "Python").is_dir() and (candidate / "test").is_dir():
            return candidate
    raise FileNotFoundError("Run from inside sauria-gather")

REPO_ROOT = find_repo_root(Path.cwd())
TEST_DIR = REPO_ROOT / "test"
VERILATOR_DIR = TEST_DIR / "verilator"
VANILLA_DIR = TEST_DIR / "stimuli_vanilla_for_gather_sram"
OUTPUT_DIR = TEST_DIR / "stimuli"

sys.path.insert(0, str(REPO_ROOT / "Python" / "src"))
import sauria_srama_overwrite_test as overwrite_test

print("REPO_ROOT:", REPO_ROOT)
print("VANILLA_DIR:", VANILLA_DIR)

REPO_ROOT: /home/henry/sauria
VANILLA_DIR: /home/henry/sauria/test/stimuli_vanilla_for_gather_sram


## 1. Verifica patch RTL

Applica `skip_initial_a.patch` dalla root della repository:

```bash
git apply skip_initial_a.patch
```

Il bit 19 rimane `stand_alone_keep_A`; il nuovo skip selettivo è il bit 26.

In [2]:
dma_text = (REPO_ROOT / "RTL/src/df_controller/sauria_dma_controller.sv").read_text()
interface_text = (REPO_ROOT / "RTL/src/df_controller/sauria_interface.sv").read_text()
assert "input skip_initial_A" in dma_text
assert "if (skip_initial_A)" in dma_text
assert "control_regs[21][26]" in interface_text
print("RTL skip-initial-A checks passed.")

RTL skip-initial-A checks passed.


## 2. Genera gli stimuli discriminanti

Prima esegui il notebook SRAM corrente fino al salvataggio degli stimuli vanilla
in `test/stimuli_vanilla_for_gather_sram`.

In [3]:
N_VALUES = 128

manifest = overwrite_test.make_test(
    in_dir=VANILLA_DIR,
    out_dir=OUTPUT_DIR,
    n_values=N_VALUES,
    n_dense_cols=N_VALUES,
    ifmap_dram_base=None,
    dense_stage_base=0x0009_0000,
    poison_fp16=0.0,
    idx_extra_beats=1,
    debug_words=4,
)
print(json.dumps(manifest, indent=2))

FileNotFoundError: /home/henry/sauria/test/stimuli_vanilla_for_gather_sram/GoldenStimuli.txt

## 3. Compila ed esegui

Il log deve mostrare due completamenti SAURIA con il gather nel mezzo.
Il checkpoint debug verifica SRAM A subito dopo il gather; il confronto DRAM
finale è il verdetto funzionale.

In [ ]:
SAURIA_VERSION = "FP16_8x16_AXI64"
compile_log = VERILATOR_DIR / "verilator_compile_skip_initial_a.log"
with compile_log.open("w") as f:
    compile_res = subprocess.run(
        ["sh", "./compile_sauria.sh", SAURIA_VERSION],
        cwd=VERILATOR_DIR, stdout=f, stderr=subprocess.STDOUT, text=True,
    )
assert compile_res.returncode == 0, f"Compilation failed: {compile_log}"

sim_log = VERILATOR_DIR / "sram_a_overwrite_test.log"
cmd = ["./Test-Sim", "+debug", f"+stim_path={OUTPUT_DIR}", "+max-cycles=4000000"]
with sim_log.open("w") as f:
    sim_res = subprocess.run(
        cmd, cwd=VERILATOR_DIR, stdout=f, stderr=subprocess.STDOUT, text=True,
    )
print(sim_log.read_text(errors="replace")[-7000:])
print("returncode:", sim_res.returncode)

In [ ]:
report = overwrite_test.analyze_log(
    sim_log, OUTPUT_DIR / "sram_a_overwrite_manifest.json"
)
print(json.dumps(report, indent=2))
assert report["debug_reads_match"], "SRAM A does not match the gather target"
assert report["benchmark_passed"] and report["success_banner"], (
    "Final output mismatch: check whether A was reloaded or inspect the DMA/core path."
)